In [ ]:
!pip  install langchain langchain-core langchain-community

In [ ]:
!pip install pypdf pymupdf  sentence-transformers chromadb

In [1]:
from langchain_core.documents import Document

In [ ]:
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)

In [ ]:
sample_doc

In [ ]:
Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [ ]:
type(sample_doc)

In [ ]:
# Load Text Data 
from langchain_community.document_loaders.text import TextLoader

text_loader = TextLoader('data/Python.txt', encoding="utf-8")

document= text_loader.load()

document

In [ ]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document


# # PDF data USING PYMUPDF LOADER
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

### Ingestion Pipeline

In [2]:
# data into documents 
import os 
from langchain_community.document_loaders.pdf import PyPDFLoader

C:\Users\Admin\AppData\Local\Temp\ipykernel_26392\998343610.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader


In [3]:
# Converted data into documents 
import os
from langchain_community.document_loaders import PyPDFLoader

def load_all_pdfs():
    folder_path = 'data/pdfs'
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):

        if filename.lower().endswith(".pdf"):
            # Create a complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print(f"Total PDFs: {num_docs}")
    print(f"All pages: {len(all_docs)}")

    return all_docs


In [4]:
 all_pdf_documents= load_all_pdfs()

Total PDFs: 2
All pages: 12


### covert data into chunks 

In [ ]:
# covert data into chunks 
!pip install langchain-text-splitters

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter # created chunks

def split_docs(documents,chunk_size=600,chunk_overlap=50):
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chuncked_docs=text_splitter.split_documents(documents)
    return chuncked_docs
    
    

In [7]:
chunks=split_docs(all_pdf_documents)

In [8]:
len(chunks)

91

### Chunks into Embeddings  

In [9]:
# Chunks into Embeddings  
from sentence_transformers import SentenceTransformer

In [10]:
# For Creating The Embeddings 
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print("loading model.", self.model_name)
        self.model = SentenceTransformer(self.model_name)  #load the model here 
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())
        


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [ ]:
embedding_manager = EmbeddingManager()

loading model. all-MiniLM-L6-v2


C:\Users\Admin\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

 ## Vector Store

In [ ]:
import chromadb
import uuid

In [ ]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

        def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

         # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())
        
    def add_documents(self,documents,embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")
            
         # store => ids, embedding, document (page content), metadata
        #for every individual chuck creating these four values 
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []
        

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            # id for document
            doc_id = f"doc_{uuid.uuid4()}" # unique id for everydocument
            ids.append(doc_id)

            #  metadata for individual chunk
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            # documents 
            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())
        
    

In [ ]:
vector_store=VectorStoreManager()

In [ ]:
# data => documents => chunks => embeddings => store in vector store

text=[doc.page_content for doc in chunks]
#  Create Embeddings 
emebedding = embedding_manager.generate_embeddings(texts)

# Store embeddings into vector store 

vector_store.add_documents(chunks, emebedding)



## Retrieval Pipeline

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self,query,top_k=5,score_threshold=0.0):
        #Convert query into embeddings
        query_embedding=self.embedding_manager.generate_embeddings([query])[0]

         # semantic search on chroma db based on Query and retun top k value 
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]

        # Result can be zero doc that way we are checking
        if results['documents'] and results['documents'][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0] # cosine distaneces

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score=1-distance
                
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("No Valid Documents Retrieved  ")
        
            

        return retrieved_docs 
        

            
            
        
        

In [ ]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [ ]:
rag_retriever.retrieve("Which scale doctors used to measure the x ray for knee")